# Практика: копирование, сортировка и присваивание

На этом занятии обсудим:

- оператор `=` и его формы: цепное присваивание, распаковку, распаковку с `*`;
- опасность копирования изменяемых объектов, внутри которых лежат другие
  изменяемые объекты (и опасность повторения таких коллекций);
- поверхностное и глубокое копирование, модуль `copy`;
- отношения порядка для списков и кортежей;
- метод `sort()` и функцию `sorted()`, параметры `key` и `reverse`.

## 1. Следствия итерируемости и оператор `=`

### Оператор `=` не копирует

Напомним: оператор `=` **не создает копию** объекта, а лишь связывает имя с
объектом. Для неизменяемых объектов это безобидно — изменить объект все равно
нельзя. Для изменяемых — источник неприятных сюрпризов: два имени указывают
на один и тот же объект, и изменение через одно имя видно через другое.

In [ ]:
original = [1, 2, 3]
alias = original  # новая ссылка на тот же список, а не копия

alias.append(4)
print(original)           # [1, 2, 3, 4] — оригинал изменился
print(alias is original)  # True

independent_copy = original.copy()  # явная копия (подробнее — в следующем разделе)
independent_copy.append(5)
print(original)  # [1, 2, 3, 4] — копия независима

То же самое происходит при передаче списка в функцию: параметр — это
еще одно имя для того же объекта, поэтому функция может изменить список
вызывающего кода.

In [ ]:
def add_zero(items: list[int]) -> None:
    items.append(0)


numbers = [1, 2, 3]
add_zero(numbers)
print(numbers)  # [1, 2, 3, 0]

### Формы оператора присваивания

Итак, `=` связывает имя с объектом и ничего не копирует — это нужно помнить
всегда. Но запись присваивания бывает разной: кроме привычного `имя = значение`
в Python есть несколько удобных форм, которые постоянно встречаются в
реальном коде. Разберем их по очереди и посмотрим, как каждая из них ведет
себя с изменяемыми объектами.

#### Цепное присваивание
Одно значение можно присвоить сразу нескольким именам:

```python
first = second = 0
```

Правая часть вычисляется **один раз**, и все имена связываются с одним и тем
же объектом. Для неизменяемых значений это удобно, а для изменяемых снова
опасно.

In [ ]:
first = second = 0
first += 1
print(first, second)  # 1 0 — числа неизменяемы, first просто перепривязался

first_list = second_list = []
first_list.append(1)
print(second_list)  # [1] — оба имени указывают на один и тот же список

first_list = []
second_list = []  # правильный способ получить два независимых списка

#### Распаковка

Слева от `=` можно перечислить несколько имен через запятую. Тогда значения
из правой части **распаковываются** и присваиваются по порядку:

```python
first, second = 1, 2
```

Здесь справа стоит кортеж `(1, 2)` (скобки необязательны). Именно поэтому
работает элегантный обмен значений без временной переменной.

In [ ]:
first, second = 1, 2
print(first, second)  # 1 2

first, second = second, first  # сначала вычисляется правая часть, потом присваивание
print(first, second)           # 2 1

#### Распаковка коллекций

Справа может стоять **любой итерируемый объект** — список, кортеж, строка,
`range`, генератор. Это прямое следствие итерируемости: Python перебирает
объект и раскладывает элементы по именам.

In [ ]:
first, second = [1, 2]
print(first, second)  # 1 2

first, second, third = "abc"
print(first, second, third)  # a b c

first, second, third = range(3)
print(first, second, third)  # 0 1 2

for name, age in [("Anna", 25), ("Boris", 19)]:  # распаковка в заголовке цикла
    print(name, age)

(x_coord, y_coord), label = (1, 2), "point"  # вложенная распаковка
print(x_coord, y_coord, label)               # 1 2 point

Количество имен слева должно в точности совпадать с числом элементов, иначе будет `ValueError`.

In [ ]:
first, second = [1, 2, 3]

#### Распаковка с `*`

Если элементов больше, чем имен, можно поставить перед одним из имен `*`.
Оно «заберет» все лишние элементы и **всегда** будет списком (даже пустым).
Звездочка может быть **только одна**. Если значение не нужно, по соглашению
его называют `_`.

In [ ]:
first, *_, last = [1, 2, 3, 4, 5]
print(first, last)  # 1 5 — середина нас не интересует

first, *rest = "python"
print(first, rest)  # p ['y', 't', 'h', 'o', 'n']

*head, last = (1, 2, 3)
print(head, last)  # [1, 2] 3

first, *middle, last = [1, 2]
print(first, middle, last)  # 1 [] 2 — для * осталось пусто, получился пустой список

## 2. Опасность копирования вложенных изменяемых объектов

Список хранит не сами значения, а **ссылки** на объекты. Поэтому, когда мы
копируем список, который внутри содержит другие изменяемые объекты (например,
другие списки), вопрос «что именно скопировалось?» становится важным.

Есть два самых частых способа сделать копию списка:

- срез: `items[:]`;
- метод `items.copy()`.

Оба создают **новый внешний список**, но элементы в нем — те же самые
объекты, что и в оригинале.

In [ ]:
matrix = [[1, 2], [3, 4]]

slice_copy = matrix[:]
method_copy = matrix.copy()

print(slice_copy is matrix)         # False — внешний список новый
print(method_copy is matrix)        # False — внешний список новый
print(slice_copy[0] is matrix[0])   # True — а вот внутренний список тот же самый
print(method_copy[0] is matrix[0])  # True

Теперь изменим что-нибудь **внутри** копии. Изменение внутреннего списка
через `method_copy` будет видно и в оригинале, и в другой копии — ведь это
один и тот же объект:

In [ ]:
method_copy[0].append(99)

print(matrix)       # [[1, 2, 99], [3, 4]]
print(method_copy)  # [[1, 2, 99], [3, 4]]
print(slice_copy)   # [[1, 2, 99], [3, 4]]

Изменения самого внешнего списка (добавление, удаление, замена элементов)
оригинал не затрагивают — внешние списки независимы. Опасны только
изменения **внутренних** изменяемых объектов:

In [ ]:
matrix = [[1, 2], [3, 4]]
method_copy = matrix.copy()

method_copy.append([5, 6])  # меняем внешний список копии
method_copy[1] = [0, 0]     # заменяем элемент копии целиком
print(matrix)               # [[1, 2], [3, 4]] — оригинал не изменился

method_copy[0][0] = 100     # меняем внутренний список
print(matrix)               # [[100, 2], [3, 4]] — изменился и оригинал!

### Опасность повторения коллекций

Та же проблема возникает при **повторении** (`*`) коллекции, элементы
которой — изменяемые объекты. Повторение не создает новые элементы, а
несколько раз кладет в новый список **ссылку на один и тот же объект**.

Для чисел, строк и других неизменяемых элементов это безопасно, а для
списков — ловушка. Классический пример — «таблица» из нулей:

In [ ]:
safe_row = [0] * 3                    # элементы — неизменяемые int, проблем нет
print(safe_row)                       # [0, 0, 0]

grid = [[0] * 3] * 3                  # три ссылки на ОДИН и тот же внутренний список
print(grid)                           # [[0, 0, 0], [0, 0, 0], [0, 0, 0]]

grid[0][0] = 1                        # хотели изменить одну ячейку...
print(grid)                           # [[1, 0, 0], [1, 0, 0], [1, 0, 0]] — изменились все строки

print(grid[0] is grid[1] is grid[2])  # True — это один объект

## 3. Поверхностное и глубокое копирование

Из примеров выше следует, что копирование бывает двух видов.

**Поверхностная (shallow) копия** — создается новый внешний объект-контейнер,
но его элементы — те же самые объекты, что и в оригинале (копируются только
ссылки на них). Именно такую копию делают срез `items[:]`, метод `copy()`,
конструктор `list(items)`.

**Глубокая (deep) копия** — создается новый контейнер, и **рекурсивно
копируются** все объекты, лежащие внутри него. В результате копия
полностью независима от оригинала.

Для обоих видов копирования в стандартной библиотеке есть модуль `copy`:

- `copy.copy(obj)` — поверхностная копия;
- `copy.deepcopy(obj)` — глубокая копия.

In [ ]:
import copy

matrix = [[1, 2], [3, 4]]

shallow_copy = copy.copy(matrix)
deep_copy = copy.deepcopy(matrix)

print(shallow_copy[0] is matrix[0])  # True — внутренние списки общие
print(deep_copy[0] is matrix[0])     # False — внутренние списки тоже скопированы

deep_copy[0].append(99)
print(matrix)                        # [[1, 2], [3, 4]] — оригинал не пострадал
print(deep_copy)                     # [[1, 2, 99], [3, 4]]

shallow_copy[0].append(77)
print(matrix)                        # [[1, 2, 77], [3, 4]] — а тут пострадал

Глубокое копирование не бесплатно: оно медленнее и расходует больше памяти,
поэтому применять его стоит там, где действительно нужна полная независимость
копии. Если внутри лежат только неизменяемые объекты (числа, строки,
кортежи из них), поверхностной копии достаточно.

### Как правильно «повторять» изменяемые элементы

Для повторения не нужно использовать `*` на изменяемых элементах. Вместо
этого стоит использовать **включение списка** (list comprehension): выражение
внутри вычисляется заново на каждой итерации, поэтому на каждой итерации
создается новый внутренний список.

In [ ]:
grid = [[0] * 3 for _ in range(3)]  # внутренний список создается заново 3 раза

grid[0][0] = 1
print(grid)                         # [[1, 0, 0], [0, 0, 0], [0, 0, 0]]

print(grid[0] is grid[1])           # False — это разные объекты

## 4. Отношения порядка для списков и кортежей

Списки можно сравнивать со списками, а кортежи — с кортежами с помощью
операторов `<`, `<=`, `>`, `>=`, `==`, `!=`. Сравнение выполняется
**лексикографически** — так же, как слова сравниваются в словаре:

1. элементы двух последовательностей сравниваются попарно слева направо;
2. как только найдена первая пара **различающихся** элементов, результат
   сравнения последовательностей — это результат сравнения этой пары;
3. если все пары равны, а одна из последовательностей закончилась, то
   **более короткая считается меньшей**;
4. если равны все элементы и длины совпадают, последовательности равны.

In [ ]:
print((1, 2, 3) < (1, 2, 4))          # True — первая различающаяся пара: 3 < 4
print((1, 5, 0) < (2, 0, 0))          # True — решает первая пара: 1 < 2
print([1, 2] < [1, 2, 0])             # True — префикс короче, значит меньше
print([] < [0])                       # True — пустой список меньше любого непустого
print([1, 2, 3] == [1, 2, 3])         # True
print([3] > [1, 100, 100])            # True — решает первая пара: 3 > 1
print([[1, 2], [3]] < [[1, 2], [4]])  # True — вложенные списки сравниваются так же

### Важные нюансы

- Элементы, которые сравниваются на одной позиции, должны быть **сравнимы
  между собой**. Иначе будет `TypeError`.
- Список и кортеж между собой на `<` и `>` **не сравниваются**, а `==` для
  них всегда `False`, даже если элементы одинаковые.

In [ ]:
print([1, 2] == (1, 2))  # False — разные типы

In [ ]:
print([1, 2] < (1, 2, 3))

In [ ]:
print([1, "a"] < [1, 2])

## 5. Сортировка: `sort()` и `sorted()`

В Python есть два способа отсортировать данные:

| | `list.sort()` | `sorted(iterable)` |
|---|---|---|
| что это | метод списка | встроенная функция |
| что сортирует | сам список (**на месте**) | любой итерируемый объект |
| что возвращает | `None` | **новый список** |
| исходные данные | изменяются | не изменяются |

Метод `sort()` возвращает `None`, поэтому типичная ошибка —
`numbers = numbers.sort()`: после нее в переменной оказывается `None`.

In [ ]:
numbers = [5, 2, 9, 1]

result = numbers.sort()
print(numbers)  # [1, 2, 5, 9] — список отсортирован на месте
print(result)   # None

In [ ]:
numbers = [5, 2, 9, 1]

sorted_numbers = sorted(numbers)
print(sorted_numbers)  # [1, 2, 5, 9]
print(numbers)         # [5, 2, 9, 1] — исходный список не изменился

# sorted() принимает любой итерируемый объект, а возвращает всегда список
print(sorted("python"))          # ['h', 'n', 'o', 'p', 't', 'y']
print(sorted((3, 1, 2)))         # [1, 2, 3]
print(sorted({"b": 2, "a": 1}))  # ['a', 'b'] — при переборе словаря берутся ключи
print(sorted(range(5, 0, -1)))   # [1, 2, 3, 4, 5]

Оба способа принимают одинаковые **именованные** параметры `key` и `reverse`.

### Параметр `reverse`

`reverse=True` сортирует в порядке **убывания**. По умолчанию
`reverse=False` — по возрастанию.

In [ ]:
numbers = [5, 2, 9, 1]

print(sorted(numbers, reverse=True))  # [9, 5, 2, 1]

numbers.sort(reverse=True)
print(numbers)  # [9, 5, 2, 1]

### Параметр `key`

`key` — это **функция одного аргумента**. Она вызывается один раз для
каждого элемента, а сортировка выполняется уже не по самим элементам, а по
**значениям, которые вернула эта функция**. Сами элементы при этом не
изменяются — в результате остаются исходные значения.

In [ ]:
words = ["banana", "kiwi", "apple", "fig", "cherry"]

print(sorted(words))                         # ['apple', 'banana', 'cherry', 'fig', 'kiwi']
print(sorted(words, key=len))                # ['fig', 'kiwi', 'apple', 'banana', 'cherry']
print(sorted(words, key=len, reverse=True))  # ['banana', 'cherry', 'apple', 'kiwi', 'fig']

print(sorted([-5, 2, -1, 4], key=abs))  # [-1, 2, 4, -5] — по модулю

Ключом может быть и **своя функция**. Например, отсортируем людей,
представленных кортежами `(имя, возраст)`, по возрасту:

In [ ]:
def get_age(person: tuple[str, int]) -> int:
    return person[1]


people = [("Anna", 25), ("Boris", 19), ("Clara", 25), ("Dmitry", 31)]

print(sorted(people, key=get_age))
# [('Boris', 19), ('Anna', 25), ('Clara', 25), ('Dmitry', 31)]

print(sorted(people, key=get_age, reverse=True))
# [('Dmitry', 31), ('Anna', 25), ('Clara', 25), ('Boris', 19)]

Обратите внимание: `Anna` и `Clara` имеют одинаковый ключ (25) и в обоих
результатах идут в исходном порядке. Сортировка в Python **устойчивая
(stable)**: элементы с равными ключами сохраняют взаимный порядок, а
`reverse=True` этот порядок тоже не переворачивает.

Пусть функция `key` возвращает **кортеж** — тогда элементы сортируются по
первому значению кортежа, при равенстве — по второму и так далее (это прямое
следствие лексикографического сравнения из предыдущего раздела):

In [ ]:
def get_length_and_word(word: str) -> tuple[int, str]:
    return len(word), word


words = ["pear", "fig", "banana", "kiwi", "apple", "cherry", "plum"]

print(sorted(words, key=len))
# ['fig', 'pear', 'kiwi', 'plum', 'apple', 'banana', 'cherry'] — равные длины в исходном порядке

print(sorted(words, key=get_length_and_word))
# ['fig', 'kiwi', 'pear', 'plum', 'apple', 'banana', 'cherry'] — при равной длине по алфавиту

Если в коллекции лежат несравнимые между собой элементы, сортировка
завершится ошибкой:

In [ ]:
sorted([3, "two", 1])

## 6. Практическая задача: объединение отрезков

**Условие.**

Дан список списков из двух элементов. Каждый вложенный список
`[start, end]` описывает отрезок на числовой прямой (`start <= end`).
Необходимо написать функцию, которая:

1. сортирует отрезки по координате начала;
2. объединяет пересекающиеся отрезки (отрезки, которые касаются друг друга
   в одной точке, также считаем пересекающимися).

Например, для `[[8, 10], [1, 3], [2, 6], [15, 18], [17, 20]]` результатом
должно быть `[[1, 6], [8, 10], [15, 20]]`.

In [ ]:
def merge_segments(segments: list[list[int]]) -> list[list[int]]:
    # ваш код
    return segments

### Проверка

In [ ]:
assert merge_segments([]) == []
assert merge_segments([[1, 4]]) == [[1, 4]]
assert merge_segments([[1, 3], [3, 5]]) == [[1, 5]]
assert merge_segments([[1, 3], [2, 4]]) == [[1, 4]]
assert merge_segments([[1, 10], [2, 3], [4, 5]]) == [[1, 10]]
assert merge_segments([[5, 6], [1, 2], [3, 4]]) == [[1, 2], [3, 4], [5, 6]]
assert merge_segments([[1, 2], [1, 5], [1, 3]]) == [[1, 5]]

print("All tests passed")

## 7. Практическое задание: слияние двух отсортированных списков

**Условие.**

Реализуйте функцию, которая принимает на вход два отсортированных
(по неубыванию) списка и возвращает новый отсортированный список, содержащий
все их значения.

In [ ]:
def merge_sorted_lists(first: list[int], second: list[int]) -> list[int]:
    # ваш код
    return first + second

### Проверка

In [ ]:
assert merge_sorted_lists([], []) == []
assert merge_sorted_lists([1, 2], []) == [1, 2]
assert merge_sorted_lists([], [1, 2]) == [1, 2]
assert merge_sorted_lists([1, 2, 3], [4, 5]) == [1, 2, 3, 4, 5]
assert merge_sorted_lists([4, 5], [1, 2, 3]) == [1, 2, 3, 4, 5]
assert merge_sorted_lists([1, 1], [1]) == [1, 1, 1]

print("All tests passed")